In [ ]:
pip install diffusers transformers accelerate torch imageio[ffmpeg] opencv-python


In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

# 1. Load the pipeline (downloads model ~5 GB)
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
)

# 2. Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


In [ ]:
# 3. Enable memory saving (optional, for GPUs with <12 GB VRAM)
pipe.enable_model_cpu_offload()  # offloads parts to CPU when not in use



In [ ]:
# 4. Your text prompt
prompt = "A panda playing a guitar on a cloud, animated style"

In [ ]:

# 5. Generate video frames (returns list of PIL images)
video_frames = pipe(
    prompt,
    num_frames=24,          # number of frames (duration ~1 sec at 24 fps)
    num_inference_steps=25, # quality vs speed
    guidance_scale=9.0,     # how closely to follow the prompt
).frames[0]                 # first (and only) batch



  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
video_frames

array([[[[0.55874634, 0.5690918 , 0.5770874 ],
         [0.41186523, 0.43139648, 0.4666443 ],
         [0.45986938, 0.4796753 , 0.5081024 ],
         ...,
         [0.8154297 , 0.8071289 , 0.83813477],
         [0.7338867 , 0.73913574, 0.76904297],
         [0.5097351 , 0.5060806 , 0.5251312 ]],

        [[0.48724365, 0.49939728, 0.5383301 ],
         [0.60546875, 0.6347656 , 0.6813965 ],
         [0.6723633 , 0.70373535, 0.72265625],
         ...,
         [0.57788086, 0.5640869 , 0.5993042 ],
         [0.3684082 , 0.35742188, 0.4008789 ],
         [0.29174805, 0.2791748 , 0.3121338 ]],

        [[0.55041504, 0.5630493 , 0.61694336],
         [0.654541  , 0.68688965, 0.71643066],
         [0.71240234, 0.7324219 , 0.73779297],
         ...,
         [0.36572266, 0.35021973, 0.37365723],
         [0.3022461 , 0.30419922, 0.31396484],
         [0.27612305, 0.27319336, 0.27661133]],

        ...,

        [[0.26733398, 0.27124023, 0.29785156],
         [0.3149414 , 0.30773926, 0.3310547 ]

In [ ]:
# 6. Export frames to a video file (e.g., MP4)
export_to_video(video_frames, "generated_video.mp4", fps=8)
print("Video saved as generated_video.mp4")   # <-- fixed missing parenthesis

Video saved as generated_video.mp4


In [ ]:
import os

file_path = "generated_video.mp4"
if os.path.exists(file_path):
    size = os.path.getsize(file_path)
    print(f"✅ File exists. Size: {size} bytes")
    if size < 10000:
        print("⚠️ File is too small – export likely failed (missing ffmpeg?)")
else:
    print("❌ File NOT found. Check the filename and current directory.")

✅ File exists. Size: 215124 bytes


In [ ]:
from IPython.display import HTML

video_path = "generated_video.mp4"
HTML(f'''
<video width="640" controls>
  <source src="{video_path}" type="video/mp4">
  Your browser does not support the video tag.
</video>
''')

In [ ]:
from IPython.display import Video
Video("generated_video.mp4", width=640, embed=True)